In [ ]:
#License: MIT
# Atom detection adapted from gduscher MLSTEM2025:
# https://github.com/gduscher/MLSTEM2025/blob/main/Day%201/5_Atom_Finding.ipynb

%matplotlib widget

import matplotlib.pylab as plt
import numpy as np

import pyTEMlib
import pyTEMlib.file_tools
import pyTEMlib.image_tools
import pyTEMlib.probe_tools
import pyTEMlib.atom_tools

import skimage.feature
import atomap.api as am


In [ ]:
# load the calibrated aligned image written by single_frame.ipynb
folder = '/path/to/folder/'
datasets = pyTEMlib.file_tools.open_file(folder + 'aligned.hf5')
aligned = datasets[list(datasets)[0]]   # averaged registered frame; carries .x.slope


In [ ]:
# ------- Input ------
#atoms_size = .065 # in nm
atoms_size = .075 # in nm
# --------------------

image = aligned

out_tags = {}
#image.metadata['experiment']= {'convergence_angle': 30, 'acceleration_voltage': 200000.}

In [ ]:
#https://github.com/gduscher/MLSTEM2025/blob/main/Day%201/5_Atom_Finding.ipynb



scale_x = image.x.slope
gauss_diameter = atoms_size/scale_x
gauss_probe = pyTEMlib.probe_tools.make_gauss(image.shape[0], image.shape[1], gauss_diameter)

print('Deconvolution of ', aligned.title)
LR_dataset = pyTEMlib.image_tools.decon_lr(image, gauss_probe, verbose=False)
datasets['Log_001'] = LR_dataset
extent = LR_dataset.get_extent([0,1])
fig, ax = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)
ax[0].imshow(image.T, extent = extent,vmax=np.median(np.array(image))+3*np.std(np.array(image)))
ax[1].imshow(LR_dataset.T, extent = extent, vmax=np.median(np.array(LR_dataset))+3*np.std(np.array(LR_dataset)));


In [ ]:

# ------- Input ------
threshold = 0.9 #usally between 0.01 and 0.9  the smaller the more atoms
atom_size = .06 #in nm  
# ----------------------


image = LR_dataset
#image = image_choice.dataset
scale_x = image.x.slope
blobs =  skimage.feature.blob_log(image, max_sigma=atom_size/scale_x, threshold=threshold)

fig1, ax = plt.subplots(1, 1,figsize=(8,7), sharex=True, sharey=True)
plt.title("blob detection ")

plt.imshow(image.T, interpolation='nearest',cmap='gray', vmax=np.median(np.array(image))+3*np.std(np.array(image)))
plt.scatter(blobs[:, 0], blobs[:, 1], c='r', s=20, alpha = .5);


In [ ]:
atoms = blobs
image = aligned
image = image-image.min()

atom_radius = 2
MaxInt = 0
MinInt = 0
maxDist = 2
sym = pyTEMlib.atom_tools.atom_refine(np.array(image), atoms, atom_radius, max_int = 0, min_int = 0, max_dist = 2)
refined_atoms = np.array(sym['atoms'])

fig, ax = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)
ax[0].imshow(image.T)
ax[0].scatter(refined_atoms[:,0],refined_atoms[:,1],  s=10, alpha = 0.3, color = 'red')
ax[0].set_title('refined atom postion')
ax[1].imshow(image.T)
ax[1].scatter(atoms[:, 0], atoms[:, 1], c='r', s=10, alpha = .3);
ax[1].set_title('blobs on image');


In [ ]:
plt.close('all')
plt.violinplot(sym['gauss_intensity'])
plt.show()

In [ ]:
#Export

#x = sublattice.x_position
#y = sublattice.y_position
#size = sublattice.pixel_size
#ellipticity = np.asarray(sublattice.ellipticity) - 1
#rot = -np.asarray(sublattice.rotation_ellipticity)
#np.save(folder+'planar_928_pyTEMlib',np.array(sym['atoms']))

In [ ]:

#import atomap

In [ ]:
#atom_positions = [atomap.atom_position.Atom_Position(x, y) for x, y in sym['atoms']]
sublattice = am.Sublattice(sym['atoms'], image=image.T)
sublattice.find_nearest_neighbors()

In [ ]:
sublattice.refine_atom_positions_using_2d_gaussian(
    percent_to_nn=0.4,   # default; fraction of NN distance used as fit mask
)

In [ ]:
sublattice.plot()

In [ ]:
i_points, i_record, p_record = sublattice.integrate_column_intensity()

In [ ]:
x = sublattice.x_position
y = sublattice.y_position
#size = sublattice.pixel_size
ellipticity = np.asarray(sublattice.ellipticity) - 1
rot = -np.asarray(sublattice.rotation_ellipticity)
#np.save(folder+'planar_928_pyTEMlib_atomap',np.array([x,y,ellipticity,rot,i_points]))
np.save(folder + 'atom_positions',np.array([x,y,ellipticity,rot,i_points]))

In [ ]:
fft_image = aligned.fft().abs()
fft_image.plot()

In [ ]:
aligned.shape